# Support Vector Machines (SVM)

A comprehensive guide to Support Vector Machines for classification and regression.

## Learning Objectives

- Understand the theory behind SVMs (maximum margin, support vectors)
- Master linear SVMs and the kernel trick
- Implement SVM classification and regression (SVR)
- Tune SVM hyperparameters (C, gamma, kernel)
- Understand when to use SVMs vs other algorithms

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.datasets import load_iris, make_classification, make_moons, make_circles
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. SVM Theory: Maximum Margin Classifier

SVMs find the hyperplane that maximizes the margin between classes.

**Key Concepts:**
- **Hyperplane**: Decision boundary separating classes
- **Margin**: Distance from hyperplane to nearest data points
- **Support Vectors**: Data points closest to the hyperplane

In [ ]:
# Create simple linearly separable data
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=100, centers=2, random_state=42, cluster_std=1.5)

# Fit SVM
svm = SVC(kernel='linear', C=1000)  # High C = hard margin
svm.fit(X, y)

# Visualize decision boundary and margins
def plot_svm_decision_boundary(model, X, y, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create mesh
    xlim = ax.get_xlim() if ax.get_xlim()[1] > ax.get_xlim()[0] else (X[:, 0].min() - 1, X[:, 0].max() + 1)
    ylim = ax.get_ylim() if ax.get_ylim()[1] > ax.get_ylim()[0] else (X[:, 1].min() - 1, X[:, 1].max() + 1)
    
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200))
    
    # Get decision function
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary and margins
    ax.contourf(xx, yy, Z, alpha=0.3, levels=[-1, 0, 1], colors=['blue', 'white', 'red'])
    ax.contour(xx, yy, Z, colors='k', levels=[-1, 0, 1], linestyles=['--', '-', '--'])
    
    # Plot data points
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k', s=50)
    
    # Highlight support vectors
    ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
               s=200, facecolors='none', edgecolors='green', linewidth=2, label='Support Vectors')
    
    ax.legend()
    return ax

fig, ax = plt.subplots(figsize=(10, 6))
plot_svm_decision_boundary(svm, X, y, ax)
ax.set_title(f'SVM Decision Boundary\n{len(svm.support_vectors_)} Support Vectors')
plt.show()

print(f"Number of support vectors: {len(svm.support_vectors_)}")
print(f"Support vector indices: {svm.support_}")

## 2. Soft Margin SVM (C Parameter)

The **C parameter** controls the trade-off between:
- **Large C**: Smaller margin, fewer misclassifications (hard margin)
- **Small C**: Larger margin, allows more misclassifications (soft margin)

In [ ]:
# Effect of C parameter
C_values = [0.01, 0.1, 1, 10, 100]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, C in zip(axes, C_values):
    svm = SVC(kernel='linear', C=C)
    svm.fit(X, y)
    plot_svm_decision_boundary(svm, X, y, ax)
    ax.set_title(f'C = {C}\n{len(svm.support_vectors_)} Support Vectors')
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.suptitle('Effect of C Parameter on SVM Decision Boundary', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. The Kernel Trick

For non-linearly separable data, SVMs use the **kernel trick** to map data to higher dimensions.

**Common Kernels:**
- `linear`: K(x, y) = x · y
- `poly`: K(x, y) = (γx · y + r)^d
- `rbf` (Gaussian): K(x, y) = exp(-γ||x - y||²)
- `sigmoid`: K(x, y) = tanh(γx · y + r)

In [ ]:
# Create non-linearly separable data
X_moons, y_moons = make_moons(n_samples=200, noise=0.15, random_state=42)
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.5, random_state=42)

# Compare kernels
kernels = ['linear', 'poly', 'rbf', 'sigmoid']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, kernel in enumerate(kernels):
    # Moons dataset
    svm = SVC(kernel=kernel, gamma='auto')
    svm.fit(X_moons, y_moons)
    
    ax = axes[0, i]
    xx, yy = np.meshgrid(np.linspace(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, 100),
                         np.linspace(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5, 100))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='coolwarm', edgecolors='k')
    ax.set_title(f'{kernel.upper()} Kernel\nMoons Data')
    
    # Circles dataset
    svm = SVC(kernel=kernel, gamma='auto')
    svm.fit(X_circles, y_circles)
    
    ax = axes[1, i]
    xx, yy = np.meshgrid(np.linspace(X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5, 100),
                         np.linspace(X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5, 100))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='coolwarm', edgecolors='k')
    ax.set_title(f'{kernel.upper()} Kernel\nCircles Data')

plt.tight_layout()
plt.show()

## 4. RBF Kernel and Gamma Parameter

The **gamma** parameter controls the influence of individual training examples:
- **Low gamma**: Far reach, smoother decision boundary
- **High gamma**: Close reach, more complex boundary (risk of overfitting)

In [ ]:
# Effect of gamma on RBF kernel
gamma_values = [0.01, 0.1, 1, 10, 100]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, gamma in zip(axes, gamma_values):
    svm = SVC(kernel='rbf', gamma=gamma, C=1)
    svm.fit(X_moons, y_moons)
    
    xx, yy = np.meshgrid(np.linspace(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, 100),
                         np.linspace(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5, 100))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='coolwarm', edgecolors='k')
    ax.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
               s=100, facecolors='none', edgecolors='green', linewidth=2)
    ax.set_title(f'gamma = {gamma}\n{len(svm.support_vectors_)} SVs')

plt.suptitle('Effect of Gamma on RBF Kernel Decision Boundary', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Feature Scaling for SVMs

SVMs are sensitive to feature scales. Always standardize features!

In [ ]:
# Demonstrate importance of scaling
from sklearn.datasets import load_wine

wine = load_wine()
X_wine, y_wine = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(X_wine, y_wine, test_size=0.3, random_state=42)

# Without scaling
svm_unscaled = SVC(kernel='rbf')
svm_unscaled.fit(X_train, y_train)
score_unscaled = svm_unscaled.score(X_test, y_test)

# With scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_scaled = SVC(kernel='rbf')
svm_scaled.fit(X_train_scaled, y_train)
score_scaled = svm_scaled.score(X_test_scaled, y_test)

print("Wine Classification Results:")
print(f"Without scaling: {score_unscaled:.3f}")
print(f"With scaling:    {score_scaled:.3f}")
print(f"\nImprovement: {(score_scaled - score_unscaled) * 100:.1f}%")

## 6. SVM Pipeline with Grid Search

In [ ]:
# Create pipeline with scaling and SVM
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])

# Parameter grid
param_grid = {
    'svm__C': [0.1, 1, 10],
    'svm__gamma': ['scale', 'auto', 0.1, 1],
    'svm__kernel': ['rbf', 'linear']
}

# Grid search
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_wine, y_wine)

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV Score: {grid_search.best_score_:.3f}")

# Results visualization
results = pd.DataFrame(grid_search.cv_results_)
results = results[['param_svm__C', 'param_svm__gamma', 'param_svm__kernel', 'mean_test_score', 'std_test_score']]
results = results.sort_values('mean_test_score', ascending=False).head(10)
print("\nTop 10 Parameter Combinations:")
print(results.to_string(index=False))

## 7. Multi-class Classification

SVMs are inherently binary classifiers. For multi-class:
- **One-vs-Rest (OvR)**: N binary classifiers (default in scikit-learn)
- **One-vs-One (OvO)**: N*(N-1)/2 binary classifiers

In [ ]:
# Multi-class classification on Iris dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Compare decision functions
svm_ovr = SVC(kernel='rbf', decision_function_shape='ovr')
svm_ovo = SVC(kernel='rbf', decision_function_shape='ovo')

svm_ovr.fit(X_train_scaled, y_train)
svm_ovo.fit(X_train_scaled, y_train)

print("Multi-class Classification Results:")
print(f"One-vs-Rest (OvR): {svm_ovr.score(X_test_scaled, y_test):.3f}")
print(f"One-vs-One (OvO):  {svm_ovo.score(X_test_scaled, y_test):.3f}")

# Confusion matrix
y_pred = svm_ovr.predict(X_test_scaled)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## 8. Support Vector Regression (SVR)

SVMs can also perform regression using the **epsilon-insensitive** loss function.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error, r2_score

# Create regression data with non-linear relationship
np.random.seed(42)
X_reg = np.sort(5 * np.random.rand(200, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + np.random.randn(200) * 0.1

# Compare different SVR kernels
kernels = ['linear', 'poly', 'rbf']
colors = ['blue', 'green', 'red']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, kernel, color in zip(axes, kernels, colors):
    svr = SVR(kernel=kernel, C=100, epsilon=0.1, gamma='auto')
    svr.fit(X_reg, y_reg)
    y_pred = svr.predict(X_reg)
    
    ax.scatter(X_reg, y_reg, color='gray', alpha=0.5, label='Data')
    ax.plot(X_reg, y_pred, color=color, linewidth=2, label=f'{kernel.upper()} SVR')
    ax.scatter(X_reg[svr.support_], y_reg[svr.support_], 
               facecolors='none', edgecolors=color, s=80, label='Support Vectors')
    ax.set_title(f'{kernel.upper()} Kernel\nR² = {r2_score(y_reg, y_pred):.3f}')
    ax.legend()

plt.suptitle('Support Vector Regression with Different Kernels', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. SVR Parameter Tuning

In [ ]:
# Effect of epsilon and C on SVR
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Effect of epsilon
epsilons = [0.01, 0.1, 0.5]
for ax, eps in zip(axes[0], epsilons):
    svr = SVR(kernel='rbf', C=100, epsilon=eps, gamma='auto')
    svr.fit(X_reg, y_reg)
    y_pred = svr.predict(X_reg)
    
    ax.scatter(X_reg, y_reg, color='gray', alpha=0.5)
    ax.plot(X_reg, y_pred, color='red', linewidth=2)
    ax.fill_between(X_reg.ravel(), y_pred - eps, y_pred + eps, alpha=0.2, color='red', label='ε-tube')
    ax.scatter(X_reg[svr.support_], y_reg[svr.support_], 
               facecolors='none', edgecolors='green', s=80)
    ax.set_title(f'epsilon = {eps}\n{len(svr.support_)} Support Vectors')
    ax.legend()

# Effect of C
C_values = [0.1, 10, 1000]
for ax, C in zip(axes[1], C_values):
    svr = SVR(kernel='rbf', C=C, epsilon=0.1, gamma='auto')
    svr.fit(X_reg, y_reg)
    y_pred = svr.predict(X_reg)
    
    ax.scatter(X_reg, y_reg, color='gray', alpha=0.5)
    ax.plot(X_reg, y_pred, color='blue', linewidth=2)
    ax.scatter(X_reg[svr.support_], y_reg[svr.support_], 
               facecolors='none', edgecolors='green', s=80)
    ax.set_title(f'C = {C}\nR² = {r2_score(y_reg, y_pred):.3f}')

plt.tight_layout()
plt.show()

## 10. Practical Example: Breast Cancer Classification

In [ ]:
from sklearn.datasets import load_breast_cancer

# Load data
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

print(f"Dataset shape: {X_cancer.shape}")
print(f"Classes: {cancer.target_names}")
print(f"Class distribution: {np.bincount(y_cancer)}")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

# Create pipeline
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', probability=True))
])

# Grid search for best parameters
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 0.001, 0.01, 0.1]
}

grid_search = GridSearchCV(svm_pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.3f}")

In [ ]:
# Evaluate best model
best_svm = grid_search.best_estimator_
y_pred = best_svm.predict(X_test)
y_proba = best_svm.predict_proba(X_test)[:, 1]

print("Test Set Results:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=cancer.target_names, yticklabels=cancer.target_names)
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC curve
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='blue', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 11. When to Use SVMs

**Advantages:**
- Effective in high-dimensional spaces
- Memory efficient (uses only support vectors)
- Versatile through different kernels
- Robust to overfitting with proper regularization

**Disadvantages:**
- Slow on large datasets (O(n²) to O(n³) complexity)
- Sensitive to feature scaling
- No probabilistic outputs by default
- Difficult to interpret (black-box model)

**Best Use Cases:**
- Text classification (high dimensions, sparse data)
- Image classification
- Bioinformatics (gene expression analysis)
- Small to medium-sized datasets

In [ ]:
# Compare SVM with other classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import time

models = {
    'SVM (RBF)': Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf'))]),
    'SVM (Linear)': Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='linear'))]),
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())])
}

results = []
for name, model in models.items():
    start = time.time()
    scores = cross_val_score(model, X_cancer, y_cancer, cv=5, scoring='accuracy')
    elapsed = time.time() - start
    results.append({
        'Model': name,
        'Mean Accuracy': scores.mean(),
        'Std': scores.std(),
        'Time (s)': elapsed
    })

results_df = pd.DataFrame(results).sort_values('Mean Accuracy', ascending=False)
print("Model Comparison on Breast Cancer Dataset:")
print(results_df.to_string(index=False))

## 12. Key Takeaways

1. **SVMs find the maximum margin hyperplane** separating classes
2. **C parameter** controls margin width vs. misclassification trade-off
3. **Kernel trick** enables non-linear decision boundaries
4. **Gamma parameter** (RBF kernel) controls model complexity
5. **Always scale features** before training SVMs
6. **Use pipelines** to prevent data leakage during cross-validation
7. **SVR** extends SVMs to regression with epsilon-insensitive loss
8. **Best for**: High-dimensional, small-to-medium sized datasets

In [ ]:
# Summary of SVM parameters
svm_params = pd.DataFrame({
    'Parameter': ['C', 'kernel', 'gamma', 'degree', 'coef0', 'epsilon (SVR)'],
    'Description': [
        'Regularization - controls margin vs error trade-off',
        'Kernel type: linear, poly, rbf, sigmoid',
        'Kernel coefficient (RBF, poly, sigmoid)',
        'Degree for polynomial kernel',
        'Independent term in poly/sigmoid kernel',
        'Epsilon tube width for SVR'
    ],
    'High Value Effect': [
        'Smaller margin, fewer errors (risk overfitting)',
        'N/A',
        'More complex boundary (risk overfitting)',
        'More complex boundary',
        'Shifts kernel function',
        'More points inside tube (simpler model)'
    ],
    'Low Value Effect': [
        'Larger margin, more errors (may underfit)',
        'N/A',
        'Smoother boundary (may underfit)',
        'Simpler boundary',
        'N/A',
        'Tighter fit (risk overfitting)'
    ]
})

print("SVM Parameter Guide:")
print(svm_params.to_string(index=False))